# Light Rider Quantum Quickstart

Build a quantum circuit with the [`lightrider`](https://pypi.org/project/lightrider/) Python SDK, check it on a free local simulator, then submit it to a Rigetti quantum backend through the Light Rider platform.

You need a Light Rider API key (`lr_...`) — create one at [platform.lightriderinc.com/settings/keys](https://platform.lightriderinc.com/settings/keys).

In [ ]:
import sys

major, minor = sys.version_info[:2]
print(f"Python {major}.{minor} detected.")
if major != 3 or minor < 10:
    print("\u26a0\ufe0f This notebook requires Python 3.10 or above. In Colab: Runtime \u2192 Change runtime type.")

In [ ]:
%pip install -q lightrider==1.3.1 requests

import requests

#@markdown Paste your Light Rider API key below, then run this cell.
api_key = "" #@param {type:"string"}
base_url = "https://platform.lightriderinc.com"

api_key = api_key.strip()
if not api_key.startswith("lr_"):
    raise ValueError("Expected a Light Rider API key beginning with 'lr_'. Create one at /settings/keys on the platform.")

session = requests.Session()
session.headers["Authorization"] = f"Bearer {api_key}"

## Build a circuit with the SDK

A Bell pair: Hadamard on qubit 0, then CNOT onto qubit 1, then measure both qubits.

In [ ]:
from lightrider import Circuit

circuit = Circuit(2, name="colab-bell")
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

print(circuit)

## Dry-run locally (free)

The SDK ships local simulators, so you can verify the circuit's physics before spending any credits. A Bell pair should give only `00` and `11`, roughly 50/50.

In [ ]:
from lightrider import get_backend

local = get_backend("statevector")
print("local counts:", local.run(circuit, shots=1000).result().get_counts())

## Pick a platform backend

Valid `backend` values for submission:

- **Mock (free, unlimited):** `rigetti-cepheus-mock`
- **Real QPU (costs credits, requires a prior purchase):** `rigetti-cepheus` — runs on the real Cepheus-1-108Q device and costs real money: **$0.002 per shot ($2 per 1,000 shots)**.

The real device is shared — if another customer holds it under a reservation, submitting a job returns a "no capacity available" error rather than queuing it. That's expected, not a bug; the error message below tells you when it's next free. For guaranteed access at a specific time, book a slot on the "Cepheus-1-108Q (Reserved)" card at [platform.lightriderinc.com/backends](https://platform.lightriderinc.com/backends) instead.

In [ ]:
backend = "rigetti-cepheus"  # change this to the backend you want to use

response = session.post(
    f"{base_url}/api/lr/quantum/submit",
    json={"backend": backend, "circuit": circuit.to_payload(), "shots": 1000},
)
if response.status_code == 402:
    # Billing gate: real QPUs need purchased credits and a sufficient balance.
    detail = response.json()
    raise RuntimeError(f"{detail['error']}: {detail['message']}")
if response.status_code == 503:
    detail = response.json()
    if detail.get("error") == "busy":
        # Expected, not a bug: someone else holds Cepheus-1-108Q under a
        # reservation right now. Retry later, or book a guaranteed window
        # on the "Cepheus-1-108Q (Reserved)" card instead.
        raise RuntimeError(
            f"Cepheus-1-108Q has no capacity right now. Next available at "
            f"{detail['nextAvailableAt']}. Try again later, or book a guaranteed "
            f"slot at platform.lightriderinc.com/backends."
        )
response.raise_for_status()
job = response.json()
print("Job submitted:", job["job_uuid"])
print("Status:", job["status"])

In [ ]:
import time

job_id = job["job_uuid"]

for _ in range(300):
    status_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}")
    status_response.raise_for_status()
    status_data = status_response.json()
    print("Status:", status_data["status"])

    if status_data.get("isInTerminalState"):
        break
    time.sleep(1)
else:
    raise TimeoutError("Job did not complete within 5 minutes. Check platform.lightriderinc.com/jobs for status.")

result_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}/result")
result_response.raise_for_status()
counts = result_response.json()["counts"]
print("Counts:", counts)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(counts.keys(), counts.values())
plt.xlabel("Measurement outcome")
plt.ylabel("Count")
plt.title(f"Bell pair on {backend}")
plt.show()

This circuit creates a **Bell pair** — two qubits entangled so they always agree when measured. On real hardware, expect the large majority of results in `00` and `11`, roughly split 50/50 — but not exclusively, unlike the exact distribution the local simulator predicted above: gate and readout noise typically leave a Bell pair on decent qubits around 90% correlated, with the rest showing up as `01`/`10`. That's normal noise, not a broken circuit — a device reading noticeably worse than that on a circuit this simple is the actual signal something's off.

## Billing and plans

Every new signup gets 1000 free Light Rider credits automatically — no payment required. `rigetti-cepheus-mock` is always free and unlimited regardless of your balance.

Switch `backend` to `rigetti-cepheus` to run on the real Cepheus-1-108Q QPU — that costs credits ($0.002 per shot, i.e. $2 per 1,000 shots), and requires having purchased credits at least once (the free signup credits alone don't unlock real hardware). If you haven't purchased yet, you'll get a 402 (`purchase_required`) instead of a job id. If you have purchased before but your balance has since run out, you'll get a 402 (`insufficient_credits`) instead. Buy credits at [/settings/purchases/quantum-compute](https://platform.lightriderinc.com/settings/purchases/quantum-compute).

Your submitted jobs — from this notebook and from the web dashboard alike — are listed at [platform.lightriderinc.com/jobs](https://platform.lightriderinc.com/jobs).